In [ ]:
import sys
import re
import ollama

from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QTextEdit,
    QPushButton, QHBoxLayout, QSizePolicy, QComboBox, QLabel, QSpacerItem
)

from PyQt5.QtCore import QThread, pyqtSignal, Qt
from PyQt5.QtGui import QFont


# ==============================
# AI WORKER THREAD
# ==============================

class AIWorker(QThread):
    token_received = pyqtSignal(str)
    finished = pyqtSignal()

    def __init__(self, messages, remember, model):
        super().__init__()
        self.messages = messages
        self.remember = remember
        self.model = model

    def run(self):
        if self.remember:
            msgs = self.messages
        else:
            msgs = [self.messages[-1]]

        response = ollama.chat(
            model=self.model,
            messages=msgs,
            stream=True
        )

        for chunk in response:
            token = chunk["message"]["content"]
            self.token_received.emit(token)

        self.finished.emit()


# ==============================
# MAIN APP
# ==============================

class ChatApp(QWidget):

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Local AI Chat")
        self.resize(1000, 850)

        self.messages = []
        self.remember_mode = True
        self.model_name = "deepseek-r1:8b"

        # ==============================
        # LIGHT UI STYLE
        # ==============================

        self.setStyleSheet("""
        QWidget {
            background-color: #b8baba;
            color: #222;
            font-family: Segoe UI;
        }

        QTextEdit {
            background-color: white;
            border-radius: 10px;
            padding: 10px;
            border:1px solid #ddd;
        }

        QPushButton {
            border-radius: 10px;
            font-weight:bold;
        }

        QPushButton:hover {
            background-color:#45a049;
        }

        QComboBox {
            background:white;
            padding:6px;
            border-radius:6px;
            border:1px solid #ccc;
        }
        """)

        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(10, 10, 10, 30)
        main_layout.setSpacing(10)

        # ==========================
        # TOP BAR: MODEL + MEMORY
        # ==========================

        # ==============================
# TOP BAR: MODEL + MEMORY (UPGRADED)
# ==============================
        
        top_layout = QHBoxLayout()
        top_layout.setSpacing(20)
        
        # Model dropdown with styled heading
        model_container = QVBoxLayout()
        model_label = QLabel("Select Model")
        model_label.setFont(QFont("Segoe UI", 11, QFont.Bold))
        model_label.setStyleSheet("color:#333;")
        model_container.addWidget(model_label)
        self.model_box = QComboBox()
        self.model_box.addItems([
            "deepseek-r1:8b",
            "deepseek-coder:6.7b",
            "deepseek-coder-v2:16b",
            "llama3.1:8b",
            "qwen2.5:14b",
            "codellama:13b",
            "gpt-oss:20b",
            "qwen3-coder:30b"
        ])
        self.model_box.setStyleSheet("""
        QComboBox {
            background:#f5f5f5;
            border:1px solid #ccc;
            border-radius:6px;
            padding:6px;
        }
        QComboBox::drop-down {
            subcontrol-origin: padding;
            subcontrol-position: top right;
            width: 20px;
            border-left-width: 1px;
            border-left-color: #ccc;
            border-left-style: solid;
            border-top-right-radius:3px;
            border-bottom-right-radius:3px;
        }
        """)
        model_container.addWidget(self.model_box)
        top_layout.addLayout(model_container)
        
        top_layout.addStretch()
        
        # Memory dropdown with styled heading
        memory_container = QVBoxLayout()
        memory_label = QLabel("Chat Memory")
        memory_label.setFont(QFont("Segoe UI", 11, QFont.Bold))
        memory_label.setStyleSheet("color:#333;")
        memory_container.addWidget(memory_label)
        self.memory_box = QComboBox()
        self.memory_box.addItems(["Memory ON", "Memory OFF"])
        self.memory_box.setStyleSheet("""
        QComboBox {
            background:#f5f5f5;
            border:1px solid #ccc;
            border-radius:6px;
            padding:6px;
        }
        QComboBox::drop-down {
            subcontrol-origin: padding;
            subcontrol-position: top right;
            width: 20px;
            border-left-width: 1px;
            border-left-color: #ccc;
            border-left-style: solid;
            border-top-right-radius:3px;
            border-bottom-right-radius:3px;
        }
        """)
        memory_container.addWidget(self.memory_box)
        top_layout.addLayout(memory_container)
        
        main_layout.addLayout(top_layout)
        # ==========================
        # CHAT DISPLAY
        # ==========================

        self.chat_display = QTextEdit()
        self.chat_display.setReadOnly(True)
        self.chat_display.setFont(QFont("Segoe UI", 13))
        chat_container = QHBoxLayout()
        
        chat_container.addSpacerItem(QSpacerItem(80, 10))
        chat_container.addWidget(self.chat_display)
        chat_container.addSpacerItem(QSpacerItem(80, 10))
        
        main_layout.addLayout(chat_container)

        # ==========================
        # BOTTOM PANEL: CENTERED PROMPT + SEND
        # ==========================

        bottom_container = QHBoxLayout()
        bottom_container.addStretch()  # left space

        bottom_layout = QHBoxLayout()
        bottom_layout.setSpacing(40)

        # PROMPT AREA (WIDER, LARGER HEIGHT)
        self.input_box = QTextEdit()
        self.input_box.setFont(QFont("Segoe UI", 13))
        self.input_box.setPlaceholderText("Type your message here...")
        self.input_box.setFixedHeight(200)
        self.input_box.setFixedWidth(2500)  # not full width
        bottom_layout.addWidget(self.input_box)

        # SEND BUTTON
        self.send_button = QPushButton("SEND ➤")
        self.send_button.setFont(QFont("Segoe", 25))
        self.send_button.setFixedHeight(130)
        self.send_button.setFixedWidth(150)
        self.send_button.setStyleSheet("""
        QPushButton{
            background-color:#4CAF50;
            color:white;
            font-size:18px;
        }
        QPushButton:hover{
            background-color:#45a049;
        }
        """)
        bottom_layout.addWidget(self.send_button)

        bottom_container.addLayout(bottom_layout)
        bottom_container.addStretch()  # right space

        # Add bottom container to main layout with bottom margin (10px from bottom)
        main_layout.addSpacerItem(QSpacerItem(20, 10))
        main_layout.addLayout(bottom_container)
        main_layout.addSpacerItem(QSpacerItem(20, 10))

        self.setLayout(main_layout)

        # ==========================
        # SIGNALS
        # ==========================

        self.send_button.clicked.connect(self.send_message)
        self.model_box.currentTextChanged.connect(self.change_model)
        self.memory_box.currentTextChanged.connect(self.change_memory)

        self.input_box.installEventFilter(self)

    # ==============================
    # ENTER TO SEND
    # ==============================

    def eventFilter(self, obj, event):
        if obj == self.input_box and event.type() == event.KeyPress:
            if event.key() == Qt.Key_Return and not event.modifiers():
                self.send_message()
                return True
        return super().eventFilter(obj, event)

    # ==============================
    # MODEL CHANGE
    # ==============================

    def change_model(self, model):
        self.model_name = model
        self.messages = []
        self.chat_display.clear()
        self.chat_display.append(f"<p style='color:blue'>Model switched to {model}</p>")

    # ==============================
    # MEMORY MODE
    # ==============================

    def change_memory(self, mode):
        if mode == "Memory ON":
            self.remember_mode = True
            self.chat_display.append("<p style='color:green'>Memory Enabled</p>")
        else:
            self.remember_mode = False
            self.chat_display.append("<p style='color:red'>Memory Disabled</p>")
   

    

    # ==============================
    # SEND MESSAGE
    # ==============================

    def send_message(self):
        user_text = self.input_box.toPlainText().strip()
        if not user_text:
            return
    
        self.input_box.clear()
    
        # show user message simply
        self.chat_display.append(f"<span style='color:#0b5394; font-weight:600'>You:</span> {user_text}<br>")
    
        self.messages.append({"role": "user", "content": user_text})
    
        self.chat_display.append("<span style='color:#444; font-weight:600'>AI:</span> ")
    
        self.worker = AIWorker(self.messages, self.remember_mode, self.model_name)
        self.worker.token_received.connect(self.update_stream)
        self.worker.finished.connect(self.finish_response)
    
        self.current_response = ""
        self.worker.start()

    # ==============================

    def update_stream(self, token):
        cursor = self.chat_display.textCursor()
        cursor.movePosition(cursor.End)
        cursor.insertText(token)
    
        self.chat_display.ensureCursorVisible()
    
        self.current_response += token
        # ==============================

    def finish_response(self):
        self.chat_display.append("\n")
    
        if self.remember_mode:
            self.messages.append({
                "role": "assistant",
                "content": self.current_response
            })


# ==============================
# RUN APP
# ==============================

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ChatApp()
    window.showMaximized()
    sys.exit(app.exec_())